# nb134 — Batched docking (fixes nb126's 12h timeout)

Previous: 6 receptors × 4,652 compounds × 12 sec/dock = 93h → cancelled at 12h with 12% coverage.

Fix:
1. **Reduce exhaustiveness 4 → 2** (~2x faster, still good for ligand-based prediction)
2. **Reduce to 3 top-diversity receptors** (1ILH=SR12813, 2O9I=T0901317, 1NRL=PCN — span PXR ligand space)
3. **Batched** — use `BATCH_IDX` env var to dock compound slice [BATCH_IDX*1165 : (BATCH_IDX+1)*1165]. Push 4 kernels (BATCH_IDX=0..3) in parallel.

Per batch: 1,165 compounds × 3 receptors × 6 sec = ~6 hours. Fits comfortably in 12h limit.

In [ ]:
import os, subprocess, sys, urllib.request, stat, time
from pathlib import Path
os.environ['PYTHONUNBUFFERED'] = '1'

# Batch slice from env var (default 0 if not set)
BATCH_IDX = 0
BATCH_SIZE = 1165
print(f'BATCH_IDX={BATCH_IDX}  BATCH_SIZE={BATCH_SIZE}  -> slice [{BATCH_IDX*BATCH_SIZE} : {(BATCH_IDX+1)*BATCH_SIZE}]')

# Install tools
subprocess.run(['apt-get', 'install', '-y', '-q', 'openbabel'], check=False, capture_output=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rdkit'], check=False)

# Vina binary
VINA_BIN = '/kaggle/working/vina'
if not Path(VINA_BIN).exists():
    urllib.request.urlretrieve(
        'https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64',
        VINA_BIN)
    os.chmod(VINA_BIN, os.stat(VINA_BIN).st_mode | stat.S_IEXEC)
r = subprocess.run([VINA_BIN, '--version'], capture_output=True, text=True)
print(f'vina: {(r.stdout + r.stderr).strip()[:80]}')

In [ ]:
# 3 top-diversity PXR receptors (vs 6 in nb126)
PDB_DIR = Path('/kaggle/working/pdbs')
PDB_DIR.mkdir(exist_ok=True)
PXR_PDBS = ['1ILH', '2O9I', '1NRL']
for pdb_id in PXR_PDBS:
    out = PDB_DIR / f'{pdb_id}.pdb'
    if not out.exists():
        urllib.request.urlretrieve(f'https://files.rcsb.org/download/{pdb_id}.pdb', out)
    print(f'  {pdb_id}: {out.stat().st_size//1024} KB')

def prepare_receptor(pdb_path, out_pdbqt):
    lines = open(pdb_path).readlines()
    keep, lig_coords = [], []
    for line in lines:
        if line.startswith('HETATM'):
            if 'HOH' in line or ' ZN ' in line or ' MG ' in line or ' NA ' in line:
                continue
            try:
                lig_coords.append((float(line[30:38]), float(line[38:46]), float(line[46:54])))
            except ValueError: pass
            continue
        if line.startswith(('ATOM','TER','END')):
            keep.append(line)
    cp = pdb_path.with_suffix('.clean.pdb')
    cp.write_text(''.join(keep))
    subprocess.run(['obabel', str(cp), '-O', str(out_pdbqt), '-xr'], check=False, capture_output=True)
    if lig_coords:
        cx = sum(c[0] for c in lig_coords)/len(lig_coords)
        cy = sum(c[1] for c in lig_coords)/len(lig_coords)
        cz = sum(c[2] for c in lig_coords)/len(lig_coords)
    else:
        cx=cy=cz=0.0
    return cx, cy, cz

receptors = {}
for pdb_id in PXR_PDBS:
    pdb = PDB_DIR / f'{pdb_id}.pdb'
    pdbqt = PDB_DIR / f'{pdb_id}.pdbqt'
    cx, cy, cz = prepare_receptor(pdb, pdbqt)
    sz = pdbqt.stat().st_size//1024 if pdbqt.exists() else 0
    print(f'  {pdb_id}: center=({cx:.1f},{cy:.1f},{cz:.1f})  pdbqt={sz}KB')
    if sz > 0:
        receptors[pdb_id] = (str(pdbqt), cx, cy, cz)
print(f'Receptors: {len(receptors)}')

In [ ]:
# Load PXR compounds (train + test); take this batch's slice
import pandas as pd, urllib.request
HF = 'https://huggingface.co/datasets/openadmet/pxr-challenge-train-test/resolve/main'
if not Path('/kaggle/working/train.csv').exists():
    urllib.request.urlretrieve(f'{HF}/pxr-challenge_TRAIN.csv', '/kaggle/working/train.csv')
    urllib.request.urlretrieve(f'{HF}/pxr-challenge_TEST_BLINDED.csv', '/kaggle/working/test.csv')
tr = pd.read_csv('/kaggle/working/train.csv')
te = pd.read_csv('/kaggle/working/test.csv')
all_compounds = pd.concat([
    tr[['Molecule Name','SMILES']].assign(split='train'),
    te[['Molecule Name','SMILES']].assign(split='test'),
], ignore_index=True).rename(columns={'Molecule Name':'name','SMILES':'smiles'}).dropna()
print(f'Total compounds: {len(all_compounds)}')

# Slice for this batch
lo = BATCH_IDX * BATCH_SIZE
hi = (BATCH_IDX + 1) * BATCH_SIZE
batch = all_compounds.iloc[lo:hi].reset_index(drop=True)
print(f'This batch: rows [{lo}, {hi}] -> {len(batch)} compounds')

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
LIG_DIR = Path('/kaggle/working/ligands')
LIG_DIR.mkdir(exist_ok=True)

def make_pdbqt(name, smi):
    safe = ''.join(c if c.isalnum() else '_' for c in str(name))
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return None
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3(); params.randomSeed = 42
    if AllChem.EmbedMolecule(mol, params) < 0: return None
    try: AllChem.MMFFOptimizeMolecule(mol, maxIters=100)
    except Exception: pass
    pdb = LIG_DIR / f'{safe}.pdb'
    pdbqt = LIG_DIR / f'{safe}.pdbqt'
    try: Chem.MolToPDBFile(mol, str(pdb))
    except Exception: return None
    subprocess.run(['obabel', str(pdb), '-O', str(pdbqt), '-h'], check=False, capture_output=True)
    return pdbqt if pdbqt.exists() and pdbqt.stat().st_size > 100 else None

import re
AFF_RE = re.compile(r'\s*1\s+(-?\d+\.\d+)')
BOX = 24.0
EXH = 2  # halved from default 8 / nb126's 4 — fast and adequate for ligand-based features

def dock(cname, smi, rec_id, rec_pdbqt, cx, cy, cz):
    lig = make_pdbqt(cname, smi)
    if lig is None: return None
    safe = ''.join(c if c.isalnum() else '_' for c in str(cname))
    out = LIG_DIR / f'{safe}_{rec_id}.out.pdbqt'
    cmd = [VINA_BIN, '--receptor', str(rec_pdbqt), '--ligand', str(lig),
           '--center_x', str(cx), '--center_y', str(cy), '--center_z', str(cz),
           '--size_x', str(BOX), '--size_y', str(BOX), '--size_z', str(BOX),
           '--num_modes', '1', '--exhaustiveness', str(EXH),
           '--out', str(out)]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=90)
    except subprocess.TimeoutExpired:
        return None
    aff = None
    for line in r.stdout.splitlines():
        m = AFF_RE.match(line)
        if m: aff = float(m.group(1)); break
    out.unlink(missing_ok=True)
    lig.unlink(missing_ok=True)
    return aff

In [ ]:
# Sanity check on first compound
row0 = batch.iloc[0]
rid0, (pq0, cx0, cy0, cz0) = list(receptors.items())[0]
demo_aff = dock(row0['name'], row0['smiles'], rid0, pq0, cx0, cy0, cz0)
print(f'Sanity dock: name={row0["name"]} rec={rid0} affinity={demo_aff}')
if demo_aff is None:
    print('WARN: sanity dock failed')

In [ ]:
# Parallel dock — use all 4 cores
from concurrent.futures import ProcessPoolExecutor
import pickle
OUT = Path('/kaggle/working/dock_results')
OUT.mkdir(exist_ok=True)

tasks = []
for _, row in batch.iterrows():
    for rid, (pq, cx, cy, cz) in receptors.items():
        tasks.append((row['name'], row['smiles'], rid, pq, cx, cy, cz))
print(f'Tasks: {len(tasks)}  ({len(batch)} compounds × {len(receptors)} receptors)')

def _dock_kw(args):
    return (args[0], args[2], dock(*args))

t0 = time.time()
results = []
with ProcessPoolExecutor(max_workers=4) as ex:
    for i, res in enumerate(ex.map(_dock_kw, tasks, chunksize=20)):
        results.append(res)
        if (i+1) % 200 == 0:
            elapsed = time.time() - t0
            eta_h = elapsed/(i+1)*(len(tasks)-i-1)/3600
            print(f'  {i+1}/{len(tasks)}  elapsed {elapsed/60:.1f}m  ETA {eta_h:.2f}h')
            with open(OUT / f'dock_batch{BATCH_IDX}_partial_{i+1}.pkl', 'wb') as f:
                pickle.dump(results, f)

df = pd.DataFrame(results, columns=['name','receptor','affinity'])
df.to_parquet(OUT / f'dock_batch{BATCH_IDX}.parquet', index=False)
print(f'\nBatch {BATCH_IDX} done: {len(df)} results in {(time.time()-t0)/60:.1f}m')
print(df.describe())